# Test random-intercepts

## Load data


### Adapted notebook from Peter. June 2026. Added INLA / DALIA comparison.

In [ ]:
# setup DALIA paths
library(here)
folder_path <- here()
print(folder_path)

result_folder_dalia = paste0(folder_path, "/../../../dalia-project/DALIA/examples/g_federated")
print(result_folder_dalia)

In [ ]:
library(confeR)
library(lme4)
library(dplyr)
library(ggplot2)
set.seed(123)

# Load nurses data (Gaussian)
data(nurses_hom, package = "confeR")
df_raw <- nurses_hom$data_ipd

subsample_hospital <- function(df, id, N) {
  site_rows <- df[df$hospital == id, ]
  if (nrow(site_rows) >= N) {
    sampled_site <- site_rows[sample(nrow(site_rows), N, replace = FALSE), ]
  } else {
    sampled_site <- site_rows
  }

  other_sites <- df[df$hospital != id, ]
  rbind(sampled_site, other_sites)
}

# We can subsample some of the hospitals to simulate low sample size scenario
# Low-powered sites should have overfitted fixed-effects, hence stronger shrinkage from random intercepts
df <- subsample_hospital(df_raw, id = 1, N = 5)
df <- subsample_hospital(df, id = 2, N = 3)
df <- subsample_hospital(df, id = 4, N = 9)

# We can reduce the number of hospitals if needed
# df <- df[as.integer(df$hospital) < 10, ]

# Print sample suze for each hospital
table(df$hospital)


## Fit models


In [ ]:
# Random intercepts model
formula <- stress ~ 0 + age + (1 | hospital)
res_re <- lmer(formula, data = df)

# Fixed intercepts model
formula <- stress ~ 0 + age + hospital
res_fe <- lm(formula, data = df)

# We can also add additional covariates in the model, for now we keep it simple
# age + gender + experience + wardtype


In [ ]:
fe <- coef(res_fe)
fe_df <- data.frame(
  hospital = as.integer(sub("^hospital", "", names(fe)[grepl("^hospital", names(fe))])),
  fe = fe[grepl("^hospital", names(fe))]
)
print(fe_df)

re <- ranef(res_re)$hospital
re_df <- data.frame(
  hospital = rownames(re),
  re = re[, "(Intercept)"]
)
print(re_df)

In [ ]:
# R-INLA fits: fixed effects and random intercepts
if (!requireNamespace("INLA", quietly = TRUE)) {
  stop("INLA package not available")
}

# Keep priors consistent across both models
inla_prior_lambda <- 0.001

# Fixed-effects model in INLA (hospital as fixed effect)
formula_inla_fe <- stress ~ 0 + age + hospital
res_inla_fe <- INLA::inla(
  formula = formula_inla_fe,
  family = "gaussian",
  data = df,
  control.fixed = list(
    mean = 0,
    prec = inla_prior_lambda,
    mean.intercept = 0,
    prec.intercept = inla_prior_lambda
  ),
  control.compute = list(dic = TRUE, waic = TRUE)
 )

# Random-intercepts model in INLA (hospital as iid random effect)
formula_inla_re <- stress ~ 0 + age + f(
  hospital,
  model = "iid",
  hyper = list(prec = list(prior = "loggamma", param = c(1.0, 0.1)))
)
res_inla_re <- INLA::inla(
  formula = formula_inla_re,
  family = "gaussian",
  data = df,
  control.fixed = list(
    mean = 0,
    prec = inla_prior_lambda,
    mean.intercept = 0,
    prec.intercept = inla_prior_lambda
  ),
  control.compute = list(dic = TRUE, waic = TRUE)
 )

# Optional quick check
res_inla_fe$dic$dic
res_inla_re$dic$dic

In [ ]:
# load dalia results 
## read in DALIA results on random effects
output_file_dalia <- file.path(result_folder_dalia, "dalia_summary_nurses_hom_site_specific_intercept.csv")

dalia_results <- read.csv(output_file_dalia)
dalia_re = dalia_results$Estimate[2:(length(dalia_results$Estimate)-2)]
head(dalia_re)

In [ ]:
# need to further investigate effect of hyperparameter prior
cbind(lmer_re=re_df$re, inla_re=res_inla_re$summary.random$hospital$mean, dalia_re=dalia_re,lm_fe=fe_df$fe, inla_fe=res_inla_fe$summary.fixed[grepl("hospital", rownames(res_inla_fe$summary.fixed)), "mean"])

## Shrinkage


In [ ]:
plot_df <- merge(fe_df, re_df, by = "hospital")
plot_df$Method <- "LM/LMER"

# Build matching INLA fixed/random hospital effects
inla_fe <- res_inla_fe$summary.fixed
inla_fe_df <- data.frame(
  hospital = rownames(inla_fe)[grepl("^hospital", rownames(inla_fe))],
  fe = inla_fe[grepl("^hospital", rownames(inla_fe)), "mean"],
  stringsAsFactors = FALSE
)
inla_fe_df$hospital <- sub("^hospital", "", inla_fe_df$hospital)

inla_re_df <- data.frame(
  hospital = as.character(res_inla_re$summary.random$hospital$ID),
  re = res_inla_re$summary.random$hospital$mean,
  stringsAsFactors = FALSE
)

plot_inla_df <- merge(inla_fe_df, inla_re_df, by = "hospital")
plot_inla_df$Method <- "INLA"

# Combine and use slight y-offset for labels
plot_all_df <- rbind(plot_df, plot_inla_df)
plot_all_df$y_label <- plot_all_df$re - 0.05 * sign(plot_all_df$re)
plot_all_df$Method <- factor(plot_all_df$Method, levels = c("LM/LMER", "INLA"))

options(repr.plot.width = 8, repr.plot.height = 8)
ggplot(plot_all_df, aes(x = fe, y = re, color = Method)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "gray50") +
  geom_segment(aes(xend = fe, yend = fe), alpha = 0.7) +
  geom_point(size = 3) +
  geom_text(aes(label = hospital, y = y_label), size = 3, show.legend = FALSE) +
  scale_color_manual(values = c("LM/LMER" = "cornflowerblue", "INLA" = "darkorange3")) +
  labs(
    x = "Fixed hospital intercept",
    y = "Random hospital intercept",
    title = "Shrinkage: LM/LMER vs INLA"
  ) +
  geom_hline(yintercept = 0) +
  coord_equal() +
  theme_minimal(base_size = 18)

We see that sites 1, 2, and 4 with the small sample sizes are shrunk strongly. Note also that sites 24 and 25 were originally constructed as outlier sites in the data.


In [ ]:
# Print the hospital intercepts
# Column "re" is what we want to recover in federated estimate with DALIA
plot_df |> dplyr::select(c("hospital", "fe", "re"))


## Predictions

Let's make predictions using the patients we previously dropped during subsampling


In [ ]:
# Data with previously dropped patients
df_diff <- df_raw[!(row.names(df_raw) %in% row.names(df)), ]
df_diff$pred_fe <- predict(res_fe, newdata = df_diff)
df_diff$pred_re <- predict(res_re, newdata = df_diff) # allow.new.levels = TRUE)

print(dim(df_diff))
head(df_diff, 20)


In [ ]:
# Save df_raw to CSV with a simple fit indicator (1 = in initial fit, NA = prediction set)
outpath <- normalizePath("../../paper/data/summarized", winslash = "/", mustWork = FALSE)
if (!dir.exists(outpath)) dir.create(outpath, recursive = TRUE)

df_raw_export <- df_raw
df_raw_export$in_initial_fit <- as.integer(row.names(df_raw_export) %in% row.names(df))
df_raw_export$in_initial_fit[df_raw_export$in_initial_fit == 0] <- NA_integer_

dataname_export <- "nurses_hom"
family_export <- "gaussian"
file_name <- file.path(outpath, paste0("data_", dataname_export, "_", family_export, "_with_fit_indicator.csv"))

write.csv(df_raw_export, file_name, row.names = FALSE)
cat("Saved CSV to:", file_name, "\n")
table(is.na(df_raw_export$in_initial_fit))

In [ ]:
df_diff$err_fe <- df_diff$stress - df_diff$pred_fe
df_diff$err_re <- df_diff$stress - df_diff$pred_re

rmse <- function(e) sqrt(mean(e^2, na.rm = TRUE))
cat("Fixed-effect RMSE:", rmse(df_diff$err_fe), "\n")
cat("Random-effect RMSE:", rmse(df_diff$err_re), "\n")



In [ ]:
# sort df by hospital id for better visualization
df <- df[order(df$hospital), ]

## make predictions with INLA models
df_INLA <- df_raw
df_INLA$observed_stress <- df_INLA$stress
# here we refit the model but with the df_diff outcome marked as NA
df_INLA$stress <- ifelse(row.names(df_INLA) %in% row.names(df_diff), NA, df_INLA$stress)
df_INLA <- df_INLA[order(df_INLA$hospital), ]
head(df_INLA, 10)

In [ ]:
# Keep priors consistent across both models
inla_prior_lambda <- 0.001

# Fixed-effects model in INLA (hospital as fixed effect)
formula_inla_fe <- stress ~ 0 + age + hospital
res_inla_fe_pred <- INLA::inla(
  formula = formula_inla_fe,
  family = "gaussian",
  data = df_INLA,
  control.fixed = list(
    mean = 0,
    prec = inla_prior_lambda,
    mean.intercept = 0,
    prec.intercept = inla_prior_lambda
  ),
  control.compute = list(dic = TRUE, waic = TRUE),
  control.inla=list(int.strategy='eb')
 )

# Random-intercepts model in INLA (hospital as iid random effect)
formula_inla_re <- stress ~ 0 + age + f(
  hospital,
  model = "iid",
  hyper = list(prec = list(prior = "loggamma", param = c(1.0, 0.1)))
)
res_inla_re_pred <- INLA::inla(
  formula = formula_inla_re,
  family = "gaussian",
  data = df_INLA,
  control.fixed = list(
    mean = 0,
    prec = inla_prior_lambda,
    mean.intercept = 0,
    prec.intercept = inla_prior_lambda
  ),
  control.compute = list(dic = TRUE, waic = TRUE)#,
  #control.inla=list(int.strategy='eb')
 )

# Optional quick check
res_inla_fe$dic$dic
res_inla_re$dic$dic


In [ ]:
# Extract predictions only for rows masked as NA during INLA fit
idx_missing <- which(is.na(df_INLA$stress))

pred_compare <- data.frame(
  row_id = idx_missing,
  hospital = df_INLA$hospital[idx_missing],
  observed_stress = df_INLA$observed_stress[idx_missing],
  pred_INLA_fe = res_inla_fe_pred$summary.fitted.values[idx_missing, "mean"],
  pred_INLA_re = res_inla_re_pred$summary.fitted.values[idx_missing, "mean"]
 )
pred_compare <- pred_compare[order(pred_compare$hospital), ]

pred_compare$err_INLA_fe <- pred_compare$observed_stress - pred_compare$pred_INLA_fe
pred_compare$err_INLA_re <- pred_compare$observed_stress - pred_compare$pred_INLA_re

rmse <- function(e) sqrt(mean(e^2, na.rm = TRUE))
cat("INLA fixed-effect RMSE (masked rows):", rmse(pred_compare$err_INLA_fe), "\n")
cat("INLA random-effect RMSE (masked rows):", rmse(pred_compare$err_INLA_re), "\n")

print(dim(pred_compare))
head(pred_compare,20)


In addition to shrinkage, another advantage of random effects is that we can also make predictions for previously unseen hospitals.


In [ ]:
## read in DALIA results on random effects
result_folder_dalia <- "/Users/lisa/Library/Mobile Documents/com~apple~CloudDocs/uni/dalia-project/DALIA/examples/g_federated"
output_file_dalia <- file.path(result_folder_dalia, "dalia_predictions_nurses_hom_site_specific_intercept.csv")

dalia_results <- read.csv(output_file_dalia)
print(dim(dalia_results))

# Insert DALIA predictions and place directly after pred_INLA_re
pred_compare$dalia_pred_re <- dalia_results$mean_predict
pred_compare <- pred_compare |>
  dplyr::relocate(dalia_pred_re, .after = pred_INLA_re)

pred_compare$err_dalia_re <- pred_compare$observed_stress - pred_compare$dalia_pred_re

cat("DALIA random-effect RMSE (masked rows):", rmse(pred_compare$err_dalia), "\n")

head(pred_compare, 10)
